**Setup.** $\{t_i,\ y_i,\ \sigma_i\}$, with $y_i$ = flux minus its mean, so every Gaussian process below has zero mean.

Use the same star as Lab 1. Every 4th night (red) is the **test** set: we fit on the black **train** visits and check our predictions on the red ones. Run it.

In [ ]:
!pip install --user celerite2 tinygp jax

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from lsst.rsp import RSPDiscovery

RA, DEC = 225.6712047, -40.2819795
DIA_OBJECT_ID = 743575318450667695
BAND = "r"

# Connect to DP2
service = RSPDiscovery("dp2").get_tap_client()

# Query forced photometry
query = f"""
SELECT
    fsodo.band,
    fsodo.psfFlux,
    fsodo.psfFluxErr,
    fsodo.psfFlux_flag,
    fsodo.pixelFlags_bad,
    fsodo.pixelFlags_saturatedCenter,
    fsodo.invalidPsfFlag,
    vis.expMidptMJD
FROM dp2.ForcedSourceOnDiaObject AS fsodo
JOIN dp2.Visit AS vis
    ON vis.visit = fsodo.visit
WHERE fsodo.diaObjectId = {DIA_OBJECT_ID}
"""

job = service.submit_job(query)
job.run()
job.wait(phases=["COMPLETED", "ERROR"])

tab = job.fetch_result().to_table()

# Save data
tab.write(f"dp2_forced_{DIA_OBJECT_ID}.ecsv", overwrite=True)

ok = (
    (np.array(tab["band"], dtype=str) == BAND)
    & (tab["psfFlux_flag"] == 0)
    & (tab["pixelFlags_bad"] == 0)
    & (tab["pixelFlags_saturatedCenter"] == 0)
    & (tab["invalidPsfFlag"] == 0)
)

mjd = np.array(tab["expMidptMJD"], dtype=float)
flux = np.array(tab["psfFlux"], dtype=float) / 1e3
flux_err = np.array(tab["psfFluxErr"], dtype=float) / 1e3

ok &= np.isfinite(mjd) & np.isfinite(flux) & np.isfinite(flux_err) & (flux_err > 0)

order = np.argsort(mjd[ok])

t_all = mjd[ok][order]
flux = flux[ok][order]
sigma_all = flux_err[ok][order]

# Relative time
t_all -= np.floor(t_all.min())


In [ ]:
plt.figure(figsize=(10, 4))
plt.errorbar(t_all, flux, yerr=sigma_all, fmt="ko", ms=3)

N = t_all.size

plt.xlabel("Time [days]")
plt.ylabel(r"Flux [$\mu$Jy]")
plt.title(f"DiaObject {DIA_OBJECT_ID} — {BAND}-band")
plt.tight_layout()
plt.show()

In [ ]:
y_all = flux - np.average(flux, weights=1/sigma_all**2)    # flux minus its weighted mean
night = np.cumsum(np.r_[0, np.diff(t_all) > 0.4])          # night number of each visit
test = night % 4 == 1                                      # every 4th night is the test set
train = ~test

t_train, y_train, sigma_train = t_all[train], y_all[train], sigma_all[train]
t_test,  y_test,  sigma_test  = t_all[test],  y_all[test],  sigma_all[test]
T = t_all.max() - t_all.min()                              # time span of the data [d]
rng = np.random.default_rng(1)

**Setup.** The period $P$, found exactly as in Lab 1 (periodogram, then a $\chi^2$ scan of a Fourier series), from the train visits only. Run it.

In [ ]:
from astropy.timeseries import LombScargle

def fourier(x, P, H):
    # Lab 1 design matrix: columns 1, sin(2 pi h x / P), cos(2 pi h x / P) for h = 1 ... H
    cols = [np.ones_like(x)]
    for h in range(1, H + 1):
        cols += [np.sin(2*np.pi*h*x/P), np.cos(2*np.pi*h*x/P)]
    return np.column_stack(cols)

def chi2_fourier(P, H=8):
    # chi^2 of the best 8-harmonic fit at period P (train visits)
    A = fourier(t_train, P, H) / sigma_train[:, None]
    w = np.linalg.lstsq(A, y_train / sigma_train, rcond=None)[0]
    return np.sum((y_train/sigma_train - A @ w)**2)

f, power = LombScargle(t_train, y_train, sigma_train).autopower(minimum_frequency=0.05, maximum_frequency=20, samples_per_peak=20)
peaks = np.where((power[1:-1] > power[:-2]) & (power[1:-1] > power[2:]))[0] + 1    # local maxima
candidates = 1 / f[peaks[np.argsort(power[peaks])[::-1][:5]]]                     # the 5 strongest peaks
P0 = min(np.r_[candidates, 2*candidates], key=chi2_fourier)                       # best one (aliases, P vs 2P)
P_grid = P0 + np.linspace(-1, 1, 2001) * P0**2 / (20*T)                           # fine scan around it
P = P_grid[np.argmin([chi2_fourier(p) for p in P_grid])]
print('P =', P, 'd')

plt.errorbar((t_train/P) % 1, y_train, sigma_train, fmt='ko', ms=3)
plt.xlabel('phase')
plt.ylabel('flux - mean [uJy]')
plt.show()

**Problem 1. The straight line as a kernel.**

$$k_{\rm lin}(t, t') = \sigma_0^2 + \sigma_1^2\, t\, t'$$

A line $f(t) = b + m\,t$ with random intercept $b\sim\mathcal N(0, \sigma_0^2)$ and random slope $m\sim\mathcal N(0, \sigma_1^2)$ is a Gaussian process with this covariance between any two times. `t` and `t_prime` are arrays of times; the function returns the matrix of $k(t_i, t'_j)$ for every pair.

In [ ]:
sigma_0 = 3 * np.std(y_train)          # prior std of the intercept
sigma_1 = 3 * np.std(y_train) / T      # prior std of the slope

def k_lin(t, t_prime):
    # k_lin(t, t') = sigma_0^2 + sigma_1^2 * t * t'      (every pair: np.multiply.outer(t, t_prime))
    return ...

**Problem 2. Predicting with a Gaussian process.**

$$K = k(\mathbf t, \mathbf t)$$
$$C = \mathrm{diag}(\boldsymbol\sigma^2)$$
$$K_* = k(\mathbf t, \mathbf t_*)$$
$$K_{**} = k(\mathbf t_*, \mathbf t_*)$$
$$\boldsymbol\mu_* = K_*^\top (K + C)^{-1}\mathbf y$$
$$\Sigma_* = K_{**} - K_*^\top (K + C)^{-1} K_*$$
$$\boldsymbol\sigma_* = \sqrt{\mathrm{diag}(\Sigma_*)}$$

The data $\mathbf y$ (at times $\mathbf t$, errors $\boldsymbol\sigma$) and the unknown light curve $\mathbf f_*$ at new times $\mathbf t_*$ are jointly Gaussian; predicting is conditioning that Gaussian on the data, exactly the formula from Lecture 1. $\boldsymbol\mu_*$ is the prediction and $\boldsymbol\sigma_*$ its error bar. 

In [ ]:
def gp_predict(k, t, y, sigma, t_star):
    # K = k(t, t)
    K = ...
    # C = diag(sigma^2)
    C = ...
    # K_star = k(t, t_star)
    K_star = ...
    # K_star_star = k(t_star, t_star)
    K_star_star = ...
    # mu_star = K_star^T (K + C)^-1 y
    mu_star = ...
    # Sigma_star = K_star_star - K_star^T (K + C)^-1 K_star
    Sigma_star = ...
    # sigma_star = sqrt(diag(Sigma_star))
    sigma_star = ...
    return mu_star, sigma_star

**Problem 3. Four kernels.**

$$\tau = t - t'$$
$$k_{\rm SE} = A^2 \exp\!\left(-\frac{\tau^2}{2\ell^2}\right)$$
$$k_{\rm M32} = A^2 \left(1 + \frac{\sqrt3\,|\tau|}{\ell}\right)\exp\!\left(-\frac{\sqrt3\,|\tau|}{\ell}\right)$$
$$k_{\rm per} = A^2 \exp\!\left(-\Gamma\,\sin^2\frac{\pi\tau}{P}\right)$$
$$k_{\rm QP} = A^2 \exp\!\left(-\frac{\tau^2}{2\ell^2} - \Gamma\,\sin^2\frac{\pi\tau}{P}\right)$$

Each kernel says how similar the flux is at two times $\tau$ apart. $A$ is the size of the variability, $\ell$ how fast the similarity fades, $P$ the period after which it comes back, $\Gamma$ how sharp the repeating shape is. (`sin^2(x)` in the comments means `np.sin(x)**2`.)

In [ ]:
def k_SE(t, t_prime, A, ell):
    # tau = t - t'
    tau = np.subtract.outer(t, t_prime)
    # k_SE = A^2 exp(-tau^2 / (2 ell^2))
    return ...

def k_M32(t, t_prime, A, ell):
    # tau = t - t'
    tau = ...
    # k_M32 = A^2 (1 + sqrt(3) |tau| / ell) exp(-sqrt(3) |tau| / ell)
    return ...

def k_per(t, t_prime, A, P, Gamma):
    # tau = t - t'
    tau = ...
    # k_per = A^2 exp(-Gamma sin^2(pi tau / P))
    return ...

def k_QP(t, t_prime, A, ell, P, Gamma):
    # tau = t - t'
    tau = ...
    # k_QP = A^2 exp(-tau^2 / (2 ell^2) - Gamma sin^2(pi tau / P))
    return ...

In [ ]:
zero, one = np.zeros(1), np.ones(1)

tau_grid = np.linspace(0, 3*P, 600)
plt.plot(tau_grid, k_SE(tau_grid, zero, 1, 0.1), label='SE, ell = 0.1 d')
plt.plot(tau_grid, k_M32(tau_grid, zero, 1, 0.1), label='Matern-3/2, ell = 0.1 d')
plt.plot(tau_grid, k_per(tau_grid, zero, 1, P, 5), label='periodic, Gamma = 5')
plt.plot(tau_grid, k_QP(tau_grid, zero, 1, 1.0, P, 5), label='quasi-periodic, ell = 1 d')
plt.legend(); 
plt.xlabel('tau [d]'); 
plt.ylabel('k(tau) / A^2')
plt.show()

**Problem 4. Drawing light curves from the prior.**

$$K = k(\mathbf t, \mathbf t) + \epsilon I$$
$$L L^\top = K$$
$$\mathbf z \sim \mathcal N(\mathbf 0, I)$$
$$\mathbf f = L\,\mathbf z$$

The Cholesky factor $L$ is a "square root" of $K$: multiplying independent unit Gaussians $\mathbf z$ by $L$ gives them exactly the covariance $K$. The tiny $\epsilon I$ only keeps the computer happy. What do SE and periodic light curves look like?

In [ ]:
def sample_prior(k, t, n_samples):
    # epsilon = 1e-6 * k(0): a tiny number added to the diagonal
    epsilon = 1e-6 * k(t[:1], t[:1])[0, 0]
    # K = k(t, t) + epsilon I
    K = ...
    # L L^T = K
    L = ...
    # z ~ N(0, I): one column of len(t) unit Gaussians per sample
    z = rng.normal(size=(len(t), n_samples))
    # f = L z
    f = ...
    return f

In [ ]:
A_guess = np.std(y_train)
t_grid = np.linspace(0, 3*P, 300)
plt.plot(t_grid, sample_prior(lambda t, t_prime: k_SE(t, t_prime, A_guess, 0.1), t_grid, 2), 'C0')
plt.plot(t_grid, sample_prior(lambda t, t_prime: k_per(t, t_prime, A_guess, P, 5), t_grid, 2), 'C1')
plt.xlabel('t [d]')
plt.ylabel('f [uJy]  (blue: SE, orange: periodic)')
plt.show()

**Problem 5. Same data, two kernels.**

$$k_1 = k_{\rm SE}\ \text{with}\ A = \mathrm{std}(\mathbf y),\ \ell = 0.1\ {\rm d}$$
$$k_2 = k_{\rm per}\ \text{with}\ A = \mathrm{std}(\mathbf y),\ P,\ \Gamma = 5$$
$$\boldsymbol\mu_{*,1},\ \boldsymbol\sigma_{*,1} = \text{gp\_predict}(k_1, \ldots),\qquad \boldsymbol\mu_{*,2},\ \boldsymbol\sigma_{*,2} = \text{gp\_predict}(k_2, \ldots)$$

Predict the test night with the most visits with both kernels (hyperparameters chosen by hand). Which kernel knows what the star does on a night it never saw?

In [ ]:
best_night = np.bincount(night[test]).argmax()      # the test night with the most visits
on = night == best_night
t_star = np.linspace(t_all[on].min() - 0.05, t_all[on].max() + 0.05, 200)

# k_1 = k_SE with A = std(y), ell = 0.1
k_1 = lambda t, t_prime: k_SE(t, t_prime, np.std(y_train), 0.1)
# k_2 = k_per with A = std(y), P, Gamma = 5
k_2 = ...

# mu_star_1, sigma_star_1 = gp_predict(k_1, t_train, y_train, sigma_train, t_star)
mu_star_1, sigma_star_1 = ...
# mu_star_2, sigma_star_2 = gp_predict(k_2, t_train, y_train, sigma_train, t_star)
mu_star_2, sigma_star_2 = ...

In [ ]:
hours = 24 * (t_star - t_star[0])
plt.plot(hours, mu_star_1, label='SE'); plt.fill_between(hours, mu_star_1 - 2*sigma_star_1, mu_star_1 + 2*sigma_star_1, alpha=0.3)
plt.plot(hours, mu_star_2, label='periodic'); plt.fill_between(hours, mu_star_2 - 2*sigma_star_2, mu_star_2 + 2*sigma_star_2, alpha=0.3)
plt.errorbar(24 * (t_all[on] - t_star[0]), y_all[on], sigma_all[on], fmt='ro', ms=3)
plt.legend()
plt.xlabel('hours into the test night')
plt.ylabel('flux - mean [uJy]')
plt.show()

**Problem 6. The marginal likelihood, with Cholesky.**

$$K = k(\mathbf t, \mathbf t)$$
$$C = \mathrm{diag}(\boldsymbol\sigma^2)$$
$$L L^\top = K + C$$
$$\mathbf z = L^{-1}\mathbf y$$
$$\text{data\_fit} = -\tfrac12\,\mathbf z^\top\mathbf z$$
$$\text{complexity} = -\sum_i \ln L_{ii}$$
$$\ln p = \text{data\_fit} + \text{complexity} - \tfrac N2 \ln 2\pi$$

$\ln p$ is the log of $\mathcal N(\mathbf y\mid\mathbf 0, K + C)$: how probable the data are under the kernel. Cholesky gives both pieces: $\mathbf z^\top\mathbf z = \mathbf y^\top(K + C)^{-1}\mathbf y$ and $\sum_i\ln L_{ii} = \tfrac12\ln|K + C|$. The data fit prefers flexible kernels; the complexity term penalises them. ($L_{ii}$ is the diagonal of $L$, $N$ the number of data.) Then scan $\ell$ of the SE kernel: where is the best $\ell$?

In [ ]:
def ln_p_parts(k, t, y, sigma):
    # K = k(t, t)
    K = ...
    # C = diag(sigma^2)
    C = ...
    # L L^T = K + C
    L = ...
    # z = L^-1 y
    z = ...
    # data_fit = -1/2 z^T z
    data_fit = ...
    # complexity = -sum_i ln(L_ii)
    complexity = ...
    # ln_p = data_fit + complexity - N/2 ln(2 pi)
    N = len(y)
    ln_p = ...
    return data_fit, complexity, ln_p

**Problem 7. The cost of Cholesky.**

$$M = k_{\rm SE}(\mathbf t, \mathbf t) + 0.01\,I$$
$$L L^\top = M,\qquad \text{cost} \propto N^3$$

Every $\ln p$ needs one Cholesky factorisation of an $N\times N$ matrix. Time it for growing $N$ (random times: the timing doesn't depend on the data). Doubling $N$ multiplies the time by 8.

In [ ]:
N_values = [250, 500, 1000, 2000, 3000]
seconds = []
for N in N_values:
    t_random = np.sort(rng.uniform(0, N/50, N))        # N random times, about 50 per day
    # M = k_SE(t, t) with A = 1, ell = 0.1, plus 0.01 I
    M = ...
    start = time.perf_counter()
    # L L^T = M
    L = ...
    seconds.append(time.perf_counter() - start)

In [ ]:
plt.loglog(N_values, seconds, 'o-', label='Cholesky')
plt.loglog(N_values, seconds[-1] * (np.array(N_values)/N_values[-1])**3, 'k--', label='proportional to N^3')
plt.legend()
plt.xlabel('N')
plt.ylabel('seconds')
plt.show()

**Problem 8. Let the data choose the hyperparameters.**

$$\sigma_{{\rm tot}, i} = \sqrt{\sigma_i^2 + s^2}$$
$$-\ln p(\mathbf y\mid\boldsymbol\theta) \ \to\ \text{minimum}$$
$$\boldsymbol\theta_{\rm SE} = (\ln A,\ \ln\ell,\ \ln s),\qquad \boldsymbol\theta_{\rm per} = (\ln A,\ \ln\Gamma,\ \ln s),\qquad \boldsymbol\theta_{\rm QP} = (\ln A,\ \ln\ell,\ \ln\Gamma,\ \ln s)$$

Each function turns $\boldsymbol\theta$ into the hyperparameters ($A = e^{\theta_0}$ and so on: fitting logarithms keeps them positive), builds the kernel, adds the jitter $s$ to the error bars as in Lab 1, and returns $-\ln p$. The next cell minimises them for you.

In [ ]:
def neg_ln_p_SE(theta):
    # A = e^theta_0,  ell = e^theta_1,  s = e^theta_2
    A, ell, s = np.exp(theta[0]), np.exp(theta[1]), np.exp(theta[2])
    # k = k_SE with A, ell
    k = lambda t, t_prime: k_SE(t, t_prime, A, ell)
    # sigma_tot = sqrt(sigma^2 + s^2)
    sigma_tot = ...
    # -ln p(y | theta)
    return ...

def neg_ln_p_per(theta):
    # A = e^theta_0,  Gamma = e^theta_1,  s = e^theta_2
    A, Gamma, s = ...
    # k = k_per with A, P, Gamma
    k = ...
    # sigma_tot = sqrt(sigma^2 + s^2)
    sigma_tot = ...
    # -ln p(y | theta)
    return ...

def neg_ln_p_QP(theta):
    # A = e^theta_0,  ell = e^theta_1,  Gamma = e^theta_2,  s = e^theta_3
    A, ell, Gamma, s = ...
    # k = k_QP with A, ell, P, Gamma
    k = ...
    # sigma_tot = sqrt(sigma^2 + s^2)
    sigma_tot = ...
    # -ln p(y | theta)
    return ...

In [ ]:
ln_A, ln_s = np.log(np.std(y_train)), np.log(0.1 * np.median(sigma_train))       # starting guesses
b_A, b_s = (ln_A - 3, ln_A + 3), (ln_s - 5, ln_s + 8)                              # allowed ranges
fit_SE  = minimize(neg_ln_p_SE,  [ln_A, np.log(0.1), ln_s], method='L-BFGS-B', bounds=[b_A, (np.log(0.003), np.log(30)), b_s])
fit_per = minimize(neg_ln_p_per, [ln_A, np.log(5), ln_s],   method='L-BFGS-B', bounds=[b_A, (np.log(0.1), np.log(200)), b_s])
fit_QP  = minimize(neg_ln_p_QP,  [fit_per.x[0], np.log(300), fit_per.x[1], fit_per.x[2]], method='L-BFGS-B',
                   bounds=[b_A, (np.log(0.5), np.log(1e5)), (np.log(0.1), np.log(200)), b_s])
print('SE:       A, ell, s        =', np.exp(fit_SE.x).round(4))
print('periodic: A, Gamma, s      =', np.exp(fit_per.x).round(4))
print('QP:       A, ell, Gamma, s =', np.exp(fit_QP.x).round(4))

A_fit, Gamma_fit, s_fit = np.exp(fit_per.x)
t_star = np.linspace(0, P, 300)                    # times covering one cycle
mu_star, sigma_star = gp_predict(lambda t, t_prime: k_per(t, t_prime, A_fit, P, Gamma_fit),
                                 t_train, y_train, np.sqrt(sigma_train**2 + s_fit**2), t_star)
plt.errorbar((t_train/P) % 1, y_train, sigma_train, fmt='ko', ms=3)
plt.plot(t_star/P, mu_star); plt.fill_between(t_star/P, mu_star - 2*sigma_star, mu_star + 2*sigma_star, alpha=0.3)
plt.xlabel('phase')
plt.ylabel('flux - mean [uJy]')
plt.show()

**Problem 9. Which kernel do the data prefer?**

$$\Delta\ln p = \ln p_{\rm model} - \ln p_{\rm SE}$$

`minimize` returns the minimum of $-\ln p$ as `fit.fun`, so $\ln p = $ `-fit.fun`. A larger $\ln p$ means the data are more probable under that kernel (the complexity penalty is built in). Differences of a few matter; hundreds are decisive. Which kernel wins, and what did the QP fit do with its $\ell$?

In [ ]:
from scipy.optimize import minimize

# Delta ln p = ln p(model) - ln p(SE)       (ln p = -fit.fun)
Delta_ln_p_SE  = 0.0
Delta_ln_p_per = ...
Delta_ln_p_QP  = ...

In [ ]:
plt.bar(['SE', 'periodic', 'QP'], [Delta_ln_p_SE, Delta_ln_p_per, Delta_ln_p_QP]); plt.axhline(0, color='k', lw=0.5)
plt.ylabel('ln p - ln p(SE)')
plt.show()

**Problem 10. The honest test: predict the test nights.**

$$\boldsymbol\mu_*,\ \boldsymbol\sigma_* = \text{gp\_predict}\big(k,\ \mathbf t_{\rm train},\ \mathbf y_{\rm train},\ \boldsymbol\sigma_{\rm tot},\ \mathbf t_{\rm test}\big)$$
$$\sigma_{{\rm pred}, i} = \sqrt{\sigma_{*,i}^2 + s^2 + \sigma_{{\rm test}, i}^2}$$

The total error of a predicted measurement adds the GP's own uncertainty $\sigma_*$, the jitter $s$, and the measurement's error bar. A perfect model puts every point on the diagonal of the plot, and honest error bars cross it about 2 times in 3. Which kernel wins, and are its error bars honest?

In [ ]:
kernels = {'SE':       (lambda t, t_prime: k_SE(t, t_prime, *np.exp(fit_SE.x[:2])),                     np.exp(fit_SE.x[-1])),
           'periodic': (lambda t, t_prime: k_per(t, t_prime, np.exp(fit_per.x[0]), P, np.exp(fit_per.x[1])), np.exp(fit_per.x[-1])),
           'QP':       (lambda t, t_prime: k_QP(t, t_prime, *np.exp(fit_QP.x[:2]), P, np.exp(fit_QP.x[2])),  np.exp(fit_QP.x[-1]))}

for name, (k, s) in kernels.items():
    # sigma_tot = sqrt(sigma_train^2 + s^2)
    sigma_tot = ...
    # mu_star, sigma_star = gp_predict(k, t_train, y_train, sigma_tot, t_test)
    mu_star, sigma_star = ...
    # sigma_pred = sqrt(sigma_star^2 + s^2 + sigma_test^2)
    sigma_pred = ...
    plt.errorbar(y_test, mu_star, sigma_pred, fmt='o', ms=3, label=name)

plt.plot([y_all.min(), y_all.max()], [y_all.min(), y_all.max()], 'k', lw=0.5); plt.legend()
plt.xlabel('observed [uJy]')
plt.ylabel('predicted [uJy]')
plt.show()

**Problem 11. The Woodbury shortcut.**

$$K = A\Lambda A^\top,\qquad \Lambda = \mathrm{diag}(\boldsymbol\lambda^2),\qquad A = \text{fourier}(\mathbf t, P, H)\ \ (N\times M)$$

$$C^{-1}\mathbf y = \mathbf y / \boldsymbol\sigma^2$$
$$C^{-1}A = A / \boldsymbol\sigma^2\quad(\text{each row } i \text{ divided by } \sigma_i^2)$$
$$\Sigma_w^{-1} = \Lambda^{-1} + A^\top C^{-1}A$$
$$\mathbf u = A^\top C^{-1}\mathbf y$$
$$\text{quad} = \mathbf y^\top C^{-1}\mathbf y - \mathbf u^\top \big(\Sigma_w^{-1}\big)^{-1}\mathbf u$$
$$\text{log\_det} = \ln\lvert\Sigma_w^{-1}\rvert + \sum_m \ln\lambda_m^2 + \sum_i \ln\sigma_i^2$$
$$\ln p = -\tfrac12\,\text{quad} - \tfrac12\,\text{log\_det} - \tfrac N2\ln 2\pi$$

When the kernel comes from $M$ weights (Lab 1's harmonics, $M = 2H + 1$), $\ln p$ only needs $M\times M$ matrices. Why: likelihood × prior = posterior × marginal likelihood, Bayes' rule written with Gaussians (Hogg, Price-Whelan & Leistedt 2020); matching the two sides gives the Woodbury identity and the determinant lemma above. ($C$ is diagonal, so $C^{-1}$ is just a division; in code `A / sigma[:, None]**2` divides each row.)

In [ ]:
def ln_p_woodbury(A, lam, sigma, y):
    # Cinv_y = y / sigma^2
    Cinv_y = ...
    # Cinv_A = A / sigma^2      (each row divided: A / sigma[:, None]**2)
    Cinv_A = ...
    # Sigma_w_inv = Lambda^-1 + A^T Cinv_A,   with Lambda^-1 = diag(1 / lam^2)
    Sigma_w_inv = ...
    # u = A^T Cinv_y
    u = ...
    # quad = y^T Cinv_y - u^T (Sigma_w_inv)^-1 u
    quad = ...
    # log_det = ln|Sigma_w_inv| + sum ln(lam^2) + sum ln(sigma^2)
    log_det = ...
    # ln_p = -1/2 quad - 1/2 log_det - N/2 ln(2 pi)
    N = len(y)
    return ...

In [ ]:
H, q = 25, 0.6                                     
lam = np.r_[np.std(y_train), np.repeat(np.sqrt(2) * np.std(y_train) * q**np.arange(H), 2)]
k_fourier = lambda t, t_prime: fourier(t, P, H) @ np.diag(lam**2) @ fourier(t_prime, P, H).T     # K = A Lambda A^T
dense = ln_p_parts(k_fourier, t_train, y_train, sigma_train)[2]
woodbury = ln_p_woodbury(fourier(t_train, P, H), lam, sigma_train, y_train)

N_values, seconds_dense, seconds_woodbury = [250, 500, 1000, 2000, 3000], [], []
for N in N_values:
    t_random = np.sort(rng.uniform(0, N/50, N)); A_random = fourier(t_random, P, H); y_random = rng.normal(size=N)
    start = time.perf_counter(); ln_p_parts(lambda t, t_prime: A_random @ np.diag(lam**2) @ A_random.T, t_random, y_random, np.ones(N))
    seconds_dense.append(time.perf_counter() - start)
    start = time.perf_counter(); ln_p_woodbury(A_random, lam, np.ones(N), y_random)
    seconds_woodbury.append(time.perf_counter() - start)
plt.loglog(N_values, seconds_dense, 'o-', label='dense (N x N)')
plt.loglog(N_values, seconds_woodbury, 'o-', label=f'Woodbury ({2*H + 1} x {2*H + 1})'); plt.legend()
plt.xlabel('N'); plt.ylabel('seconds')
plt.show()

**Problem 12. celerite: a sum of oscillators.**

$$k(\tau) = \sum_{h=1}^{J} k_{\rm SHO}\big(\tau;\ \sigma_h,\ \rho_h,\ Q_h\big)$$
$$\sigma_h = \sigma_1\, q^{\,h-1}$$
$$\rho_h = P / h$$
$$Q_h = h\,Q$$

celerite needs kernels built from damped exponentials and cosines, and then computes $\ln p$ in a time proportional to $N$. The physical one is the SHO, a damped oscillator kicked by noise, with period $\rho$, amplitude $\sigma$ and quality factor $Q$ (it stays coherent for $\tau_{\rm coh} = Q\rho/\pi$). A repeating, non-sinusoidal light curve is a sum of SHOs at the harmonics $P, P/2, \ldots$: the GP version of Lab 1's Fourier series. $Q_h = hQ$ gives every harmonic the same coherence time.

In [ ]:
import celerite2
from celerite2 import terms

def sho_sum(P, sigma_1, q, Q, J=8):
    kernel = None
    for h in range(1, J + 1):
        # sigma_h = sigma_1 q^(h-1)
        sigma_h = ...
        # rho_h = P / h
        rho_h = ...
        # Q_h = h Q
        Q_h = ...
        term = terms.SHOTerm(sigma=sigma_h, rho=rho_h, Q=Q_h)
        kernel = term if kernel is None else kernel + term
    return kernel

In [ ]:
kernel = sho_sum(P, np.std(y_train), 0.6, 200.0)
gp = celerite2.GaussianProcess(kernel, mean=0.0)
gp.compute(t_train, yerr=sigma_train)
dense = ln_p_parts(lambda t, t_prime: kernel.get_value(np.subtract.outer(t, t_prime)), t_train, y_train, sigma_train)[2]

N_celerite, seconds_celerite = [1000, 3000, 10000, 30000, 100000], []
for N in N_celerite:
    t_random = np.sort(rng.uniform(0, N/50, N)); y_random = rng.normal(size=N)
    start = time.perf_counter(); gp.compute(t_random, yerr=1.0); gp.log_likelihood(y_random)
    seconds_celerite.append(time.perf_counter() - start)
plt.loglog(N_values, seconds_dense, 'o-', label='dense Cholesky')
plt.loglog(N_celerite, seconds_celerite, 'o-', label='celerite'); plt.legend()
plt.xlabel('N'); plt.ylabel('seconds per ln p')
plt.show()

**Problem 13. Fit the oscillators.**

$$\boldsymbol\theta = (\ln\sigma_1,\ q,\ \ln Q,\ \ln s,\ m)$$
$$\mathbf y \sim \mathcal N\big(m,\ K_{\rm SHO} + C\big),\qquad C = \mathrm{diag}(\boldsymbol\sigma^2 + s^2)$$
$$-\ln p(\mathbf y\mid\boldsymbol\theta)\ \to\ \text{minimum},\qquad \tau_{\rm coh} = \frac{QP}{\pi}$$

A sum of oscillators averages to zero over a cycle, and the weighted mean we subtracted is not exactly the star's cycle-averaged flux, so the mean $m$ is fitted too. Then compare with the periodic GP from Problem 8 on the test night of Problem 5. What does the fitted $Q$ say?

In [ ]:
def neg_ln_p_celerite(theta):
    # sigma_1 = e^theta_0,  q = theta_1,  Q = e^theta_2,  s = e^theta_3,  m = theta_4
    sigma_1, q, Q, s, m = ...
    # kernel = sho_sum(P, sigma_1, q, Q)
    kernel = ...
    # a celerite GP with mean m
    gp = celerite2.GaussianProcess(kernel, mean=m)
    # C = diag(sigma^2 + s^2)
    gp.compute(t_train, diag=...)
    # -ln p(y | theta)
    return -gp.log_likelihood(y_train)

In [ ]:
fit_cel = minimize(neg_ln_p_celerite, [np.log(np.sqrt(2)*np.std(y_train)), 0.6, np.log(1000), ln_s, 0.0], method='L-BFGS-B',
                   bounds=[b_A, (0.05, 0.95), (np.log(1), np.log(1e5)), b_s, (-np.std(y_train), np.std(y_train))])
sigma_1, q, Q, s, m = np.exp(fit_cel.x[0]), fit_cel.x[1], np.exp(fit_cel.x[2]), np.exp(fit_cel.x[3]), fit_cel.x[4]
at_edge = " (at the edge of its range)" if fit_cel.x[2] > np.log(1e5) - 1e-3 else ""
print(f'Q = {Q:.0f}{at_edge}    tau_coh = Q P / pi = {Q*P/np.pi:.0f} d')

gp = celerite2.GaussianProcess(sho_sum(P, sigma_1, q, Q), mean=m)
gp.compute(t_train, diag=sigma_train**2 + s**2)
t_star = np.linspace(t_all[on].min() - 0.05, t_all[on].max() + 0.05, 200)
mu_cel = gp.predict(y_train, t=t_star)
mu_per, sigma_per = gp_predict(kernels['periodic'][0], t_train, y_train, np.sqrt(sigma_train**2 + kernels['periodic'][1]**2), t_star)
hours = 24 * (t_star - t_star[0])
plt.plot(hours, mu_per, label='periodic (dense)'); plt.fill_between(hours, mu_per - 2*sigma_per, mu_per + 2*sigma_per, alpha=0.3)
plt.plot(hours, mu_cel, '--', label='sum of SHOs (celerite)')
plt.errorbar(24 * (t_all[on] - t_star[0]), y_all[on], sigma_all[on], fmt='ro', ms=3)
plt.legend()
plt.xlabel('hours into the test night'); plt.ylabel('flux - mean [uJy]')
plt.show()

**Problem 14. tinygp.**

$$k_{\rm QP} = A^2\cdot\text{ExpSquared}(\ell)\cdot\text{ExpSineSquared}(P, \Gamma)$$
$$\frac{\partial\ln p}{\partial\theta_j} = \tfrac12\,\boldsymbol\alpha^\top\frac{\partial K}{\partial\theta_j}\boldsymbol\alpha - \tfrac12\,\mathrm{tr}\!\Big[(K + C)^{-1}\frac{\partial K}{\partial\theta_j}\Big],\qquad \boldsymbol\alpha = (K + C)^{-1}\mathbf y$$

Optimisers work much better with the gradient of $\ln p$. You never code the formula above: tinygp builds kernels from pieces (`ExpSquared` is $e^{-\tau^2/2\ell^2}$, `ExpSineSquared` is $e^{-\Gamma\sin^2(\pi\tau/P)}$) and JAX differentiates the code exactly. Use `jnp` (JAX's numpy) instead of `np` inside the function. Then compare how fast the optimiser gets to the maximum with and without the gradient.

In [ ]:
import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp
from tinygp import GaussianProcess, kernels as tinygp_kernels

def neg_ln_p_tinygp(theta):
    # A = e^theta_0,  ell = e^theta_1,  Gamma = e^theta_2,  s = e^theta_3      (use jnp.exp)
    A, ell, Gamma, s = ...
    # k_QP = A^2 * ExpSquared(ell) * ExpSineSquared(P, Gamma)
    k = A**2 * tinygp_kernels.ExpSquared(scale=ell) * tinygp_kernels.ExpSineSquared(scale=P, gamma=Gamma)
    # a tinygp GP with C = diag(sigma^2 + s^2)
    gp = GaussianProcess(k, t_train, diag=...)
    # -ln p(y | theta)
    return -gp.log_probability(y_train)

In [ ]:
value_and_gradient = jax.jit(jax.value_and_grad(neg_ln_p_tinygp))
value_only = jax.jit(neg_ln_p_tinygp)
history_gradient, history_no_gradient = [], []
def with_gradient(theta):
    v, g = value_and_gradient(theta); history_gradient.append(float(v)); return float(v), np.asarray(g)
def without_gradient(theta):
    v = value_only(theta); history_no_gradient.append(float(v)); return float(v)

theta_start = np.array([ln_A, np.log(30), np.log(5), ln_s])
bounds = [b_A, (np.log(0.5), np.log(1e5)), (np.log(0.1), np.log(200)), b_s]
minimize(with_gradient, theta_start, jac=True, method='L-BFGS-B', bounds=bounds)
minimize(without_gradient, theta_start, method='L-BFGS-B', bounds=bounds)
best = min(history_gradient + history_no_gradient)
plt.semilogy(np.array(history_gradient) - best + 1, label='JAX gradient')
plt.semilogy(np.array(history_no_gradient) - best + 1, label='finite differences')
plt.legend(); plt.xlabel('likelihood evaluations'); plt.ylabel('-ln p - min + 1')
plt.show()